# Autoregressive single-AP response panel

This notebook rebuilds one paper figure panel from a CM2-NeuronalSignal viewer cache. The workflow is intentionally linear: configure paths and plotting constants, load cached neurons, apply the tracked Default Profile filters, validate AR(2) parameters, prepare plot-ready arrays, and export one SVG.

If `data/serve/Y-corr85-pnr12.json` exists, its Region and QC filters are applied. Otherwise, all cached neurons are used. The SVG is written to `notebook/autoregressive.svg` when the notebook is run from the repository root.


## Configuration

All figure dimensions, axis ranges, density ranges, and output paths live here so later edits do not require hunting through plotting code.


In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.collections as mcollections
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import SVG, display
from matplotlib.ticker import FuncFormatter

CACHE_FOLDER_CANDIDATE = Path("data/cache/Y-corr85-pnr12")
DEFAULT_PROFILE_PATH_CANDIDATE = Path("data/serve/Y-corr85-pnr12.json")
FIGURE_PATH_CANDIDATE = Path("notebook/autoregressive.svg")

FIGSIZE_IN = (5.5, 5.5)
MAIN_X_LIMIT_MS = (0.0, 3000.0)
MAIN_X_TICKS_MS = [0, 1000, 2000, 3000]
MAIN_Y_LIMIT = (0.0, 1.0)
MAIN_Y_TICKS = [0, 0.25, 0.5, 0.75, 1.0]
TRACE_SAMPLE_COUNT = 601

T_PEAK_RANGE_MS = (0.0, 400.0)
T_HALF_RANGE_MS = (400.0, 1200.0)
DENSITY_BINS = 72
DENSITY_DISPLAY_SCALE = 1e3

INSET_SPECS = [
    {
        "name": "t_peak",
        "range_ms": T_PEAK_RANGE_MS,
        "bounds": [0.48, 0.70, 0.22, 0.20],
        "xlabel": "t_peak (ms)",
        "show_ylabel": True,
    },
    {
        "name": "t_half",
        "range_ms": T_HALF_RANGE_MS,
        "bounds": [0.74, 0.70, 0.22, 0.20],
        "xlabel": "decay t_1/2 (ms)",
        "show_ylabel": False,
    },
]

COLORS = {
    "individual_trace": (0.0, 0.0, 0.0, 0.014),
    "guide": "#8A9099",
    "grid": "#E2E5E9",
    "text": "#25282C",
    "axis": "#6B7280",
    "inset_axis": "#AEB6C2",
    "tick": "#5B6470",
    "density_line": "#2F343A",
}


def resolve_cache_folder(candidate: Path) -> Path:
    for folder in (candidate, Path("..") / candidate):
        if folder.exists():
            return folder.resolve()
    raise FileNotFoundError(f"Could not find cache folder from {candidate!s}")


def resolve_figure_path(candidate: Path) -> Path:
    if candidate.parent.exists():
        return candidate
    return Path(candidate.name)


def resolve_optional_file(candidate: Path) -> Path | None:
    for path in (candidate, Path("..") / candidate):
        if path.is_file():
            return path.resolve()
    return None


CACHE_FOLDER = resolve_cache_folder(CACHE_FOLDER_CANDIDATE)
DEFAULT_PROFILE_PATH = resolve_optional_file(DEFAULT_PROFILE_PATH_CANDIDATE)
FIGURE_PATH = resolve_figure_path(FIGURE_PATH_CANDIDATE)

print(f"Cache folder: {CACHE_FOLDER}")
print(f"Default Profile: {DEFAULT_PROFILE_PATH or 'not found; using all neurons'}")
print(f"Figure path:  {FIGURE_PATH.resolve()}")


## Load Cached Neurons

Read `metadata.json`, `point.json`, and the optional tracked Default Profile. This cell only loads data; it does not filter or transform AR parameters.


In [ ]:
def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


metadata = read_json(CACHE_FOLDER / "metadata.json")
points = read_json(CACHE_FOLDER / "point.json")

ids = np.asarray(points["id"], dtype=np.int64)
x = np.asarray(points["x"], dtype=np.float64)
y = np.asarray(points["y"], dtype=np.float64)
metrics = {key: np.asarray(value, dtype=np.float64) for key, value in points["metrics"].items()}
frame_rate_hz = float(metadata["frame_rate_hz"])

ui_state = read_json(DEFAULT_PROFILE_PATH) if DEFAULT_PROFILE_PATH else None

print(f"Neuron count in cache: {len(ids)}")
print(f"Frame rate: {frame_rate_hz:g} Hz")
print(f"Default Profile: {'loaded' if ui_state else 'not found; using all neurons'}")


## Apply Viewer Filters

Reproduce the viewer's raw-value QC range filters and Region polygon filters. Null or absent QC endpoints are unbounded; active ranges are lower-inclusive and upper-exclusive. Non-finite metric samples follow the viewer's QC coercion to zero without changing the analysis arrays, and Region polygon edges are inclusive. The selected population used in the main panel and inset distributions is `selected_mask`.


In [ ]:
def point_on_segment(px: float, py: float, ax: float, ay: float, bx: float, by: float) -> bool:
    dx = bx - ax
    dy = by - ay
    cross = (px - ax) * dy - (py - ay) * dx
    scale = max(abs(dx), abs(dy), 1.0)
    if abs(cross) > 1e-6 * scale:
        return False
    dot = (px - ax) * dx + (py - ay) * dy
    if dot < -1e-6:
        return False
    return dot <= dx * dx + dy * dy + 1e-6


def point_in_polygon(px: float, py: float, polygon: list[dict[str, float]]) -> bool:
    inside = False
    j = len(polygon) - 1
    for i, point in enumerate(polygon):
        xi = float(point["x"])
        yi = float(point["y"])
        xj = float(polygon[j]["x"])
        yj = float(polygon[j]["y"])
        if point_on_segment(px, py, xi, yi, xj, yj):
            return True
        if ((yi > py) != (yj > py)) and (px < (xj - xi) * (py - yi) / (yj - yi) + xi):
            inside = not inside
        j = i
    return inside


def metric_filter_mask(state: dict | None, metric_values: dict[str, np.ndarray]) -> np.ndarray:
    neuron_count = len(next(iter(metric_values.values())))
    mask = np.ones(neuron_count, dtype=bool)
    if not state:
        return mask

    for key, range_state in (state.get("qcRanges") or {}).items():
        if key not in metric_values or not isinstance(range_state, dict):
            continue
        lower_value = range_state.get("lower")
        upper_value = range_state.get("upper")
        if lower_value is None and upper_value is None:
            continue

        lower = -np.inf if lower_value is None else float(lower_value)
        upper = np.inf if upper_value is None else float(upper_value)
        if lower > upper:
            lower, upper = upper, lower

        raw_values = np.asarray(metric_values[key], dtype=np.float64)
        values = np.where(np.isfinite(raw_values), raw_values, 0.0)
        mask &= (values >= lower) & (values < upper)
        print(f"QC {key}: raw=[{lower:.6g}, {upper:.6g})")
    return mask


def region_filter_mask(state: dict | None, x_values: np.ndarray, y_values: np.ndarray) -> np.ndarray:
    if not state:
        return np.ones(len(x_values), dtype=bool)
    polygons = state.get("regionPolygons") or []
    if not polygons:
        return np.ones(len(x_values), dtype=bool)
    return np.asarray(
        [any(point_in_polygon(px, py, polygon) for polygon in polygons) for px, py in zip(x_values, y_values, strict=True)],
        dtype=bool,
    )


metric_mask = metric_filter_mask(ui_state, metrics)
region_mask = region_filter_mask(ui_state, x, y)
selected_mask = metric_mask & region_mask

print(f"After QC filters:      {int(metric_mask.sum())}")
print(f"After Region filters:  {int(region_mask.sum())}")
print(f"After Region + QC:     {int(selected_mask.sum())}")


## Convert AR(2) Parameters

CNMF-E stores `g` as AR parameters for the fluorescence impulse response. For AR(2), the characteristic roots are converted to fast/slow continuous time constants, then the peak-normalized kernel is `exp(-t / tau_slow) - exp(-t / tau_fast)`.

The cached `t_half` values used below are decay half-times after each neuron's own peak, not absolute times from stimulus onset.


In [ ]:
def roots_from_g(g0: float, g1: float) -> tuple[float, float]:
    disc = g0 * g0 + 4.0 * g1
    if disc <= 0.0:
        raise ValueError(f"Expected positive discriminant, got {disc}")
    sqrt_disc = math.sqrt(disc)
    r1 = 0.5 * (g0 + sqrt_disc)
    r2 = 0.5 * (g0 - sqrt_disc)
    if not (0.0 < r1 < 1.0 and 0.0 < r2 < 1.0):
        raise ValueError(f"Unexpected roots: r1={r1}, r2={r2}")
    return r1, r2


def tau_from_root(root: float, dt_s: float) -> float:
    return -dt_s / math.log(root)


def kernel_value(t_s: float, tau_fast_s: float, tau_slow_s: float) -> float:
    return math.exp(-t_s / tau_slow_s) - math.exp(-t_s / tau_fast_s)


def kernel_shape(t_s: np.ndarray, tau_fast_s: float, tau_slow_s: float) -> np.ndarray:
    values = np.exp(-t_s / tau_slow_s) - np.exp(-t_s / tau_fast_s)
    values[values < 0.0] = 0.0
    peak = float(np.max(values))
    if peak <= 0.0:
        raise ValueError("Kernel peak must be positive")
    return values / peak


def t_peak_continuous(tau_fast_s: float, tau_slow_s: float) -> float:
    return tau_fast_s * tau_slow_s / (tau_slow_s - tau_fast_s) * math.log(tau_slow_s / tau_fast_s)


def half_decay_after_peak(tau_fast_s: float, tau_slow_s: float, t_peak_s: float) -> tuple[float, float]:
    peak_value = kernel_value(t_peak_s, tau_fast_s, tau_slow_s)
    if peak_value <= 0.0:
        raise RuntimeError("Kernel peak must be positive")
    target = 0.5 * peak_value
    lo = t_peak_s
    hi = t_peak_s + 20.0 * tau_slow_s
    while kernel_value(hi, tau_fast_s, tau_slow_s) > target:
        hi += 10.0 * tau_slow_s
        if hi > t_peak_s + 200.0 * tau_slow_s:
            raise RuntimeError("Failed to bracket half-decay crossing")
    for _ in range(120):
        mid = 0.5 * (lo + hi)
        if kernel_value(mid, tau_fast_s, tau_slow_s) > target:
            lo = mid
        else:
            hi = mid
    t_abs = 0.5 * (lo + hi)
    return t_abs - t_peak_s, t_abs


def timing_from_g(g0: float, g1: float, dt_s: float) -> tuple[float, float, float, float]:
    r1, r2 = roots_from_g(g0, g1)
    tau1_s = tau_from_root(r1, dt_s)
    tau2_s = tau_from_root(r2, dt_s)
    tau_fast_s, tau_slow_s = min(tau1_s, tau2_s), max(tau1_s, tau2_s)
    peak_s = t_peak_continuous(tau_fast_s, tau_slow_s)
    half_after_peak_s, _ = half_decay_after_peak(tau_fast_s, tau_slow_s, peak_s)
    return tau_fast_s, tau_slow_s, peak_s, half_after_peak_s


def valid_ar_mask(base_mask: np.ndarray, metric_values: dict[str, np.ndarray]) -> np.ndarray:
    g0 = metric_values["g_0"]
    g1 = metric_values["g_1"]
    tp = metric_values["t_peak"]
    th = metric_values["t_half"]
    finite = base_mask & np.isfinite(g0) & np.isfinite(g1) & np.isfinite(tp) & np.isfinite(th)
    valid = finite.copy()
    for idx in np.flatnonzero(finite):
        try:
            roots_from_g(float(g0[idx]), float(g1[idx]))
        except ValueError:
            valid[idx] = False
    return valid


dt_s = 1.0 / frame_rate_hz
ar_mask = valid_ar_mask(selected_mask, metrics)
selected_indices = np.flatnonzero(selected_mask)
ar_indices = np.flatnonzero(ar_mask)

print(f"Selected neurons: {selected_indices.size}")
print(f"Valid AR(2) timing neurons: {ar_indices.size}")


## Prepare Plot Data

Build Illustrator-friendly line segments for the main response panel and normalized density curves for the inset distributions. No figure styling happens in this cell.


In [ ]:
if ar_indices.size == 0:
    raise RuntimeError("No valid selected AR(2) neurons to plot")


g0_values = metrics["g_0"][ar_indices]
g1_values = metrics["g_1"][ar_indices]
timing_values_ms = {
    "t_peak": metrics["t_peak"][ar_indices],
    "t_half": metrics["t_half"][ar_indices],
}

line_t_s = np.linspace(MAIN_X_LIMIT_MS[0] / 1000.0, MAIN_X_LIMIT_MS[1] / 1000.0, TRACE_SAMPLE_COUNT)
line_x_ms = line_t_s * 1000.0


def build_kernel_segments(g0_array: np.ndarray, g1_array: np.ndarray) -> list[np.ndarray]:
    segments = []
    for g0, g1 in zip(g0_array, g1_array, strict=True):
        try:
            tau_fast_s, tau_slow_s, _, _ = timing_from_g(float(g0), float(g1), dt_s)
            y_line = kernel_shape(line_t_s, tau_fast_s, tau_slow_s)
        except (ValueError, RuntimeError):
            continue
        segments.append(np.column_stack((line_x_ms, y_line)).astype(np.float32, copy=False))
    return segments


def smooth_density(values: np.ndarray, x_range: tuple[float, float], bins: int = DENSITY_BINS) -> tuple[np.ndarray, np.ndarray]:
    x_min, x_max = x_range
    finite = np.asarray(values, dtype=np.float64)
    finite = finite[np.isfinite(finite)]
    hist, edges = np.histogram(finite, bins=bins, range=(x_min, x_max), density=True)

    sigma = 1.4
    radius = 5
    kernel_x = np.arange(-radius, radius + 1, dtype=np.float64)
    kernel = np.exp(-0.5 * (kernel_x / sigma) ** 2)
    kernel /= np.sum(kernel)

    smooth = np.convolve(hist, kernel, mode="same")
    centers = 0.5 * (edges[:-1] + edges[1:])
    area = float(np.trapezoid(smooth, centers))
    if area > 0.0 and np.isfinite(area):
        smooth = smooth / area
    return centers, smooth


segments = build_kernel_segments(g0_values, g1_values)
if not segments:
    raise RuntimeError("No valid AR(2) kernels could be drawn")

distributions = {}
for spec in INSET_SPECS:
    xs, density = smooth_density(timing_values_ms[spec["name"]], spec["range_ms"])
    distributions[spec["name"]] = {"x": xs, "density": density}

density_y_max = 1.10 * max(float(np.nanmax(item["density"])) for item in distributions.values())
if not np.isfinite(density_y_max) or density_y_max <= 0.0:
    density_y_max = 1.0

print(f"Drawable kernel traces: {len(segments)}")
print(f"Density y-limit: {density_y_max * DENSITY_DISPLAY_SCALE:.3g} x 10^-3/ms")


## Render And Export SVG

This cell contains only figure styling, layout, and export. The main trace collection is given the SVG group id `neuron_traces` so the individual paths stay easy to select in Illustrator.


In [ ]:
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "axes.labelsize": 10,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "svg.fonttype": "none",
    }
)


def add_distribution_inset(
    parent_ax,
    xs: np.ndarray,
    density: np.ndarray,
    spec: dict,
    y_max: float,
) -> None:
    x_min, x_max = spec["range_ms"]
    inset = parent_ax.inset_axes(spec["bounds"])
    inset.set_zorder(8)
    inset.set_facecolor((1.0, 1.0, 1.0, 0.86))
    inset.fill_between(xs, 0.0, density, color="black", alpha=0.10, linewidth=0)
    inset.plot(xs, density, color=COLORS["density_line"], linewidth=0.95)
    inset.set_xlim(x_min, x_max)
    inset.set_ylim(0.0, y_max)
    inset.set_xlabel(spec["xlabel"], fontsize=6.7, color=COLORS["text"], labelpad=1.0)
    inset.set_xticks([x_min, 0.5 * (x_min + x_max), x_max])
    inset.set_xticklabels([f"{x_min:.0f}", f"{0.5 * (x_min + x_max):.0f}", f"{x_max:.0f}"])
    inset.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y * DENSITY_DISPLAY_SCALE:g}"))
    if spec["show_ylabel"]:
        inset.set_ylabel(r"Density ($10^{-3}$/ms)", fontsize=6.7, color=COLORS["text"], labelpad=1.5)
    else:
        inset.tick_params(labelleft=False)
    inset.tick_params(axis="both", labelsize=5.8, colors=COLORS["tick"], length=2.0, width=0.55, pad=1.0)
    for side in ("top", "right"):
        inset.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        inset.spines[side].set_color(COLORS["inset_axis"])
        inset.spines[side].set_linewidth(0.65)


fig, ax = plt.subplots(figsize=FIGSIZE_IN, constrained_layout=True)

background_collection = mcollections.LineCollection(
    segments,
    colors=COLORS["individual_trace"],
    linewidths=0.28,
    clip_on=False,
    zorder=1,
)
background_collection.set_gid("neuron_traces")
ax.add_collection(background_collection, autolim=False)
ax.axhline(0.5, color=COLORS["guide"], alpha=0.70, linestyle=":", linewidth=0.85, zorder=2)

for spec in INSET_SPECS:
    distribution = distributions[spec["name"]]
    add_distribution_inset(ax, distribution["x"], distribution["density"], spec, density_y_max)

ax.set_xlim(*MAIN_X_LIMIT_MS)
ax.set_xticks(MAIN_X_TICKS_MS)
ax.set_ylim(*MAIN_Y_LIMIT)
ax.set_yticks(MAIN_Y_TICKS)
ax.set_box_aspect(1)
ax.set_xlabel("Time after 1 AP (ms)")
ax.set_ylabel("Peak-normalized response")
ax.grid(color=COLORS["grid"], alpha=0.70, linewidth=0.50)
ax.spines["left"].set_color(COLORS["axis"])
ax.spines["bottom"].set_color(COLORS["axis"])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_PATH)
print(f"Saved figure to {FIGURE_PATH.resolve()}")
# display(SVG(filename=str(FIGURE_PATH)))

plt.close(fig)